#### LCEL (Langchain expression language)

In [1]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
import sys
sys.path.append("..")
import config

listInstructions = CommaSeparatedListOutputParser().get_format_instructions()

In [2]:
chatTemplate = ChatPromptTemplate.from_messages([("human", # HumanMessagePromptTemplate
                                                  "I've recently adopted a {pet}. Could you suggest three {pet} names \n" + listInstructions)])
chatTemplate

ChatPromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template="I've recently adopted a {pet}. Could you suggest three {pet} names \nYour response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`"), additional_kwargs={})])

In [3]:
chat = ChatOpenAI(
    model = "openai/gpt-4",
    seed = 365,
    api_key=config.apiKey2,
    temperature = 0,
    base_url="https://openrouter.ai/api/v1",
    max_tokens = 100
)

In [4]:
listOutputParser = CommaSeparatedListOutputParser()

In [5]:
chatTempResult = chatTemplate.invoke({"pet": "dog"})

In [6]:
res = chat.invoke(chatTempResult)

In [7]:
listOutputParser.invoke(res)

['Max', 'Bella', 'Charlie']

In [8]:
chain = chatTemplate | chat | listOutputParser

In [9]:
chain.invoke({"pet": "dog"})

['Max', 'Bella', 'Luna']

In [10]:
chain.batch([{"pet":"dog"}, {"pet": "giraffe"}])

[['Max', 'Bella', 'Luna'], ['Spot', 'Stretch', 'Tallulah']]

In [24]:
res = chain.stream({"pet": "dog"})
res

<generator object RunnableSequence.stream at 0x0000018FA4EAB5A0>

In [49]:
#next(res)

In [25]:
for i in res:
    print(i, end =  "")

['Max']['Bella']['Luna']

In [26]:
type(chain)

langchain_core.runnables.base.RunnableSequence

In [28]:
type(chatTemplate)

langchain_core.prompts.chat.ChatPromptTemplate

In [29]:
# Prompt => dictionary => PromptValue
# ChatModel => string, list or PromptValue => ChatMessage
# OutputParser => ChatMessage => parsed output
# Retriever => string => list of docs
# Tool => string or dictionary => tool output

In [33]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough # passes inputs through without alteration, identity function

In [34]:
RunnablePassthrough().invoke("Hello world")

'Hello world'

#### Pipe chains together

In [36]:
chatTemplateTools = ChatPromptTemplate.from_template("""
What are the five most important tools a {job} needs?
Answer only by listing the tools.
""")
chatTemplateStrategy = ChatPromptTemplate.from_template("""
Considering the tools provided, develop a strategy for effectively mastering them:
{tools}
""")

In [38]:
parser = StrOutputParser()

In [39]:
chatTemplateTools

ChatPromptTemplate(input_variables=['job'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job'], input_types={}, partial_variables={}, template='\nWhat are the five most important tools a {job} needs?\nAnswer only by listing the tools.\n'), additional_kwargs={})])

In [44]:
chainTools = chatTemplateTools | chat | parser | {"tools": RunnablePassthrough()}
chainStrategy = chatTemplateStrategy | chat | parser

In [46]:
chainFinal = chainTools | chainStrategy

In [48]:
print(chainFinal.invoke({"job": "fullstack developer"}))

1. Integrated Development Environment (IDE): Start by understanding the basic functionalities of the IDE such as creating a new project, writing code, debugging, and running the program. Gradually, explore advanced features like code refactoring, version control integration, and database management. Practice by creating small projects and gradually move to complex ones. 

2. Version Control System (Git): Start by understanding the importance of version control in software development. Learn the basic commands like git add, git commit, git push,


#### Visualizing chains

In [51]:
chainFinal.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
  +--------------------+   
  | ChatPromptTemplate |   
  +--------------------+   
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
     +-------------+       
     | Passthrough |       
     +-------------+       
            *              
            *              
            *       

#### RunnableParallel

In [52]:
from langchain_core.runnables import RunnableParallel

In [53]:
chatTemplateBooks = ChatPromptTemplate.from_template("""
   Suggest three of the best advanced-level {programmingLanguage} books.
   Answer only by listing the books.
""")

chatTemplateProjects = ChatPromptTemplate.from_template("""
   Suggest three interesting {programmingLanguage} projects suitable for advanced-level programmers.
   Answer only by listing the projects.
""")

In [54]:
chainBooks = chatTemplateBooks | chat | parser
chainProjects = chatTemplateProjects | chat | parser

In [56]:
chainParallel = RunnableParallel({"books": chainBooks, "projects": chainProjects})

In [58]:
chainParallel.invoke({"programmingLanguage": "C++"})

{'books': '1. "Effective Modern C++: 42 Specific Ways to Improve Your Use of C++11 and C++14" by Scott Meyers\n2. "The C++ Programming Language" by Bjarne Stroustrup\n3. "C++ Concurrency in Action" by Anthony Williams',
 'projects': '1. Building a Multiplayer Online Game Server\n2. Developing a Machine Learning Algorithm Library\n3. Creating a High-Performance Database Management System'}

In [61]:
%%time
chainParallel.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            

In [63]:
chatTemplateTime = ChatPromptTemplate.from_template("""
    I'm an advanced level programmer.
    Consider the following literature:
    {books}
    Also, consider the following projects?
    {projects}
    Roughly how much time would it take me to complete the literature and the projects?
""") 

In [65]:
chat2 = ChatOpenAI(
    model = "openai/gpt-4",
    seed = 365,
    api_key=config.apiKey2,
    temperature = 0,
    base_url="https://openrouter.ai/api/v1",
    max_tokens = 500
)

In [69]:
chainTime = RunnableParallel({"books":chainBooks, "projects": chainProjects}) | chatTemplateTime | chat2 | parser
# same as {"books":chainBooks, "projects": chainProjects}) | chatTemplateTime | chat2 | parser

In [67]:
print(chainTime.invoke({"programmingLanguage": "Javascript"}))

The time it takes to complete the literature and the projects can vary greatly depending on several factors such as your reading speed, comprehension level, familiarity with the topics, and the amount of time you can dedicate each day. 

However, as a rough estimate:

1. "JavaScript: The Good Parts" by Douglas Crockford: This book is around 170 pages long. If you read and comprehend around 20 pages a day, it would take approximately 8-9 days.

2. "You Don't Know JS" (series) by Kyle Simpson: This is a series of 6 books, each around 100-200 pages long. If you read one book a week, it would take approximately 6 weeks.

3. "Eloquent JavaScript: A Modern Introduction to Programming" by Marijn Haverbeke: This book is around 450 pages long. If you read and comprehend around 20 pages a day, it would take approximately 22-23 days.

For the projects:

1. Building a Real-Time Chat Application with Node.js and Socket.io: Depending on your familiarity with Node.js and Socket.io, this could take an

chainTime.get_graph().print_ascii()

#### RunnableLambda

In [73]:
findSum = lambda x: sum(x)
findSq = lambda x:x **2
# cannot be piped into a chain

In [84]:
from langchain_core.runnables import RunnableLambda # convert any function into a Runnable

In [75]:
runnableSum = RunnableLambda(lambda x: sum(x))

In [76]:
runnableSum.invoke([1,2,5])

8

In [80]:
runnableSq =  RunnableLambda(lambda x:x **2)
runnableSq.invoke(8)

64

In [81]:
chain = runnableSum | runnableSq

In [82]:
chain.invoke([1,2,5])

64

In [83]:
chain.get_graph().print_ascii()

+-------------+  
| LambdaInput |  
+-------------+  
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
   +--------+    
   | Lambda |    
   +--------+    
        *        
        *        
        *        
+--------------+ 
| LambdaOutput | 
+--------------+ 


#### @chain decorator

In [87]:
from langchain_core.runnables import chain as coreChain
@coreChain
def runnableSum(x):
    return sum(x)

@coreChain
def runnableSq(x):
    return x**2

In [88]:
type(runnableSq)

langchain_core.runnables.base.RunnableLambda